In [1]:
import siibra
from nilearn import plotting, image
import matplotlib.pyplot as plt
import numpy as np
import nibabel as nib
import plotly.graph_objects as go

import pyNN.neuron as sim
from pyNN import space
from pyNN.random import RandomDistribution, NumpyRNG
from pyNN.neuron import Projection, StaticSynapse, FixedNumberPreConnector
import pandas as pd
from scipy.interpolate import LinearNDInterpolator, NearestNDInterpolator
import statistics

[siibra:INFO] Version: 1.0.1-alpha.10
[siibra:WARNING] This is a development release. Use at your own risk.
[siibra:INFO] Please file bugs and issues at https://github.com/FZJ-INM1-BDA/siibra-python.
/home/tkmcg/neuron-env/lib/python3.12/site-packages/pyNN/neuron/__init__.py:14: UserWarning: mpi4py not available
  warnings.warn("mpi4py not available")


## Define the sphere class

In [2]:
class Sphere2():
    def __init__(self, radius):
        self.radius = radius
                
    def generate_positions(self, n):
        """Return `n` points distributed randomly with uniform density within the sphere with encapsulation layer"""
        num_points = n
        
        x_points = [] 
        y_points = [] 
        z_points = [] 
        
        theta = np.random.uniform(0, 2 * np.pi, num_points) 
        phi = np.random.uniform(0, np.pi, num_points) 
        x_radius = self.radius*np.cbrt(np.random.uniform(0, 1, num_points))  
        y_radius = self.radius*np.cbrt(np.random.uniform(0, 1, num_points))
        z_radius = self.radius*np.cbrt(np.random.uniform(0, 1, num_points))
            
        x_points = x_radius * np.sin(phi) * np.cos(theta)
        y_points = y_radius * np.sin(phi) * np.sin(theta) 
        z_points = z_radius * np.cos(phi)

        invalid_x = (x_points > -500) & (x_points < 500)
        invalid_y = (y_points > -1500) & (y_points < 2000)
        
        invalid = invalid_x & invalid_y

        while np.any(invalid):
            n = invalid.sum()

            # Sample spherical distribution
            theta = np.random.uniform(0, 2*np.pi, n)
            phi   = np.random.uniform(0, np.pi, n)
            r     = 2000 * np.cbrt(np.random.uniform(0, 1, n))

            # Compute new x and y for all invalid points at once
            new_x = r * np.sin(phi) * np.cos(theta)
            new_y = r * np.sin(phi) * np.sin(theta)

            # Assign
            x_points[invalid] = new_x
            y_points[invalid] = new_y

            # Recompute masks
            invalid_x = (x_points > -500) & (x_points < 500)
            invalid_y = (y_points > -1500) & (y_points < 2000)
            invalid = invalid_x & invalid_y

        points = np.column_stack((x_points, y_points, z_points))

            
        return points

## Caoimhe's Head Model

In [12]:
voltage_file = pd.read_csv(
    "../MDT3389_Mesh2_050126_E3_BSGrid.txt",
    sep=r"\s+",
    header=None,
    names=["x", "y", "z", "V"]
)

#translate the center to the center we have already identified
x_voltage_coordinates = voltage_file['x'].values*1000
y_voltage_coordinates = voltage_file['y'].values*1000
z_voltage_coordinates = voltage_file['z'].values*1000-6250

#convert to mV since NEURON and PyNN use mV
voltages = voltage_file['V'].values*1000

points = np.array([x_voltage_coordinates,y_voltage_coordinates,z_voltage_coordinates]).T

print(min(points[1]))

-3000.0


In [37]:
 Vint = LinearNDInterpolator(points, voltages)

## Boston Scientific

In [4]:
voltage_file = pd.read_csv(
    "../2202_Mesh0_E1_270625_BSGrid_V.txt",
    sep=r"\s+",
    header=None,
    names=["x", "y", "z", "V"]
)

#voltage_file = voltage_file[voltage_file['V'].notna()]

#translate the center to the center we have already identified
x_voltage_coordinates = voltage_file['x'].values*1000
y_voltage_coordinates = voltage_file['y'].values*1000
z_voltage_coordinates = voltage_file['z'].values*1000

#convert to mV since NEURON and PyNN use mV
voltages = voltage_file['V'].values*1000

points = np.array([x_voltage_coordinates,y_voltage_coordinates,z_voltage_coordinates]).T

print(points)

[[-5000.             0.         -6750.        ]
 [-5000.           500.         -6750.        ]
 [-5000.          1000.         -6750.        ]
 ...
 [ 7580.12701892  3129.1651246  22250.        ]
 [ 7830.12701892  3562.17782649 22250.        ]
 [ 8080.12701892  3995.19052838 22250.        ]]


In [50]:
 Vint = LinearNDInterpolator(points, voltages)

## Create a simple cell model with extracellular input

In [51]:
from nrnutils import Mechanism, Section

class SimpleNeuron(object):

    def __init__(self, **parameters):
        hh = Mechanism('hh', gl=parameters['g_leak'], el=-65, gnabar=parameters['gnabar'], gkbar=parameters['gkbar'])
        self.soma = Section(L=30, diam=30, mechanisms=[hh])
        self.soma.add_synapse('ampa', 'Exp2Syn', e=0.0, tau1=0.1, tau2=5.0)
        self.initial_values = {"v": -65.0}

        self.soma.insert("extracellular")

        # needed for PyNN
        self.source_section = self.soma
        self.source = self.soma(0.5)._ref_v
        self.parameter_names = ('g_leak', 'gnabar', 'gkbar')
        self.traces = {}
        self.recording_time = False

In [52]:
from pyNN.neuron import NativeCellType

class SimpleNeuronType(NativeCellType):
    default_parameters = {'g_leak': 0.0002, 'gkbar': 0.036, 'gnabar': 0.12}
    default_initial_values = {'v': -65.0}
    recordable = ['soma(0.5).v', 'soma(0.5).ina']
    units = {'soma(0.5).v' : 'mV', 'soma(0.5).ina': 'nA'}
    receptor_types = ['soma.ampa']
    model = SimpleNeuron

## Make the populations

In [45]:
##### Add in the STN neurons and change the geometry to match the coupling project with DC current ####
import pyNN.neuron as sim
import matplotlib.pyplot as plt
from pyNN import space
from pyNN.random import RandomDistribution, NumpyRNG
import numpy as np
from pyNN.neuron import Projection, StaticSynapse, FixedNumberPreConnector
import pandas as pd
from scipy.interpolate import LinearNDInterpolator, NearestNDInterpolator

v_init = -68
rng_seed = 856
n = 100
parcellation = "JULICH_BRAIN_CYTOARCHITECTONIC_ATLAS_V3_1" 
region = "STN (Subthalamus) right"
space = "MNI_152_ICBM_2009C_NONLINEAR_ASYMMETRIC"
stimulating_electrode = np.array([11600,-16200,-9100])/1000
encapsulation = 1.135


############ Setup the simulation##########
sim.setup(timestep=0.01)

########## create a population of neurons ###############
cortical_pop = sim.Population(n, SimpleNeuronType())


random_point_generator = Sphere2(2000)
STN_space = random_point_generator.generate_positions(n)
cortical_pop.positions[0], cortical_pop.positions[1],cortical_pop.positions[2] = STN_space[:,0], STN_space[:,1], STN_space[:,2]

print(cortical_pop.positions[0])

[ -662.33861354 -1430.45833594   -68.12721253   569.97673037
  1210.40845257  1279.55572914  -587.9777776  -1607.09713168
 -1220.43209406  1689.52803068 -1530.94164783  1069.93911519
  -808.3429115   -912.21763509  -574.66127515  -822.63650401
  -596.73419255  1831.33442581 -1189.94279917   563.65345466
  1158.14480717  1447.26103556 -1221.96870881  1060.29436562
  -693.36570586  -683.03589492   777.19542334 -1214.5654832
  1851.62893389 -1453.39711607   790.20894586    60.26973756
  1128.53280097 -1191.87038708 -1515.75793522  -731.83737407
   623.44415637   886.19739445   910.75353563 -1001.14440947
  1583.05837839  1186.87836679   530.65874615  1033.19811587
   650.50462439  -650.63241477  1145.80661052  -581.76155453
 -1295.88599117 -1916.05565399 -1256.61969129 -1120.48280513
  1764.27408497 -1162.58420041  1297.45159015  -725.84531572
   602.17042453  1141.13050739    -3.07606144  -473.60418439
 -1148.74577696  1690.90071114   751.21080939  -664.60362641
  -676.17640391   740.406

## Add in the positions

In [41]:
x_points_ctx = cortical_pop.positions[0]
y_points_ctx = cortical_pop.positions[1]
z_points_ctx = cortical_pop.positions[2]
interpolated_values_ctx = Vint(np.column_stack((x_points_ctx, y_points_ctx, z_points_ctx)))
efield = np.column_stack((x_points_ctx, y_points_ctx, z_points_ctx,interpolated_values_ctx))

In [42]:
for cell in cortical_pop:
    test = (cell.position == efield[:, :-1])
    indices = np.where(np.all(test, axis=1))[0]
    indices = int(indices[0])
    cell.e_extracellular = efield[indices, 3]
    print(cell, cell.position, indices, cell.e_extracellular)

0 [-1639.75698435   441.18442679  -205.83596145] 0 nan
1 [1649.83771224  219.19799384  615.89699008] 1 nan
2 [-1551.9362717   1034.30504127  -752.9751232 ] 2 nan
3 [ 731.38580534 -623.0405451  1471.40782304] 3 nan
4 [ 983.88019878 -494.40914754 1086.67590059] 4 nan
5 [-802.35354823 1449.21996365 -440.78469611] 5 nan
6 [1565.50045826  861.87730667 1664.99592232] 6 nan
7 [-1395.23009823   647.94642208   584.5196464 ] 7 nan
8 [-1189.8263551    -59.39393678 -1523.09252444] 8 nan
9 [-904.48801419 1313.30792763  614.31284855] 9 nan
10 [ -567.11165389  -439.70100573 -1276.91372547] 10 nan
11 [-1143.94018436 -1184.77388411   790.1114257 ] 11 nan
12 [ 729.2819065   599.86157206 -240.86363655] 12 nan
13 [ -698.32532825  -401.25542777 -1167.42660976] 13 nan
14 [-1400.58045997  -937.00699186  -605.00730254] 14 nan
15 [-558.05635165  510.22355684 -975.36490628] 15 nan
16 [ 751.19090027 1155.47268773  764.82422517] 16 nan
17 [-650.96370922  500.64680922 1560.00427119] 17 nan
18 [ 1084.09972284  -611